# Chatnik dev

Anton Antonov  
April 2026

----

## Sequence diagram

In [ ]:
my $flowchart = slurp($*CWD ~ /../README.md);
$flowchart = do with $flowchart.match(/ ^^ '```mermaid' (.*?) '```' $$ /) { $0.Str }

In [ ]:
llm-prompt('MermaidDiagram')

In [ ]:
# my $res = llm-synthesize([
#     llm-prompt('MermaidDiagram')($flowchart, 'sequence diagram'),
# ])

```mermaid
sequenceDiagram
    participant CCommand as Chat command
    participant IngestCODB as Chat objects file ingestion
    participant CODBOS as Chat objects file
    participant CODB as Chat objects
    participant CIDQ as Chat ID specified?
    participant CIDEQ as Chat ID exists in DB?
    participant RECO as Retrieve existing chat object
    participant PromParse as Prompt DSL spec parsing
    participant KPFQ as Known prompts found?
    participant PromExp as Prompt expansion
    participant COEval as Message evaluation
    participant CCommandOutput as Chat result
    participant CNCO as Create new chat object
    participant CIDNone as Assume chat ID is NONE
    participant UpdateCODB as Chat objects file update
    participant LLMFunc as LLM Functions
    participant LLMProm as LLM Prompts

    CCommand->>IngestCODB: Chat command
    CODBOS--)IngestCODB: Chat objects file
    IngestCODB--)CODB: Chat objects
    IngestCODB->>CIDQ: Chat ID specified?
    CIDQ-->>CIDEQ: Yes
    CIDQ-->>CIDNone: No
    CIDNone->>CIDEQ: Assume chat ID is NONE
    CIDEQ-->>RECO: Yes
    CIDEQ-->>CNCO: No
    CIDEQ--)CODB: Chat objects
    RECO->>PromParse: Prompt DSL spec parsing
    PromParse--)LLMProm: LLM Prompts
    CNCO--)LLMFunc: LLM Functions
    CNCO--)CODB: Chat objects
    CNCO->>PromParse: Prompt DSL spec parsing
    PromParse->>KPFQ: Known prompts found?
    KPFQ-->>PromExp: Yes
    KPFQ-->>COEval: No
    PromExp--)LLMProm: LLM Prompts
    PromExp->>COEval: Message evaluation
    COEval--)LLMFunc: LLM evaluator invocation
    LLMFunc--)COEval: Evaluation result
    COEval->>UpdateCODB: Chat objects file update
    COEval->>CCommandOutput: Chat result
```

---

## LLM prompts

In [3]:
use LLM::Tooling;

sub-info(&llm-prompt-data)

{arity => 0, count => Inf, description => Get the prompts database as hash with the keys being the prompt titles. C<$name> -- Str:D or Regex used to retrieve the prompts by name. C<$fields> -- Fields to provide in the result., name => llm-prompt-data, parameters => ({default => (Any), description => , name => , named => False, optional => False, position => 0, slurpy => False, type => (Any)}), required => [], returns => (Hash)}

In [7]:
llm-prompt-data(/MermaidDiagram/, fields => <Name Description PositionalArguments NamedArguments>):pairs

{MermaidDiagram => {Description => Gives Mermaid-JS diagram code over text, Name => MermaidDiagram, NamedArguments => {}, PositionalArguments => {$txt => , $type => mind-map}}}

In [3]:
llm-prompt('CopyEdit')('TEXT', format => 'HTML')

Perform basic copy editing on the following HTML text, correcting errors in grammar, spelling and punctuation; improvements to style and clarity may also be made, but do not make more significant changes to content or structure: 
TEXT

---

## CLI unit tests

In [5]:
#% bash
llm-chat -i=mh --prompt=@MadHatter 'Hi, who are you?'

Why, tick-tock and twiddle-twirl! I am the Mad Hatter, the silliest sipper of the tea-time tapestry! A dash of tea leaves here, a sprinkle of whimsy there, and voilà—madness in a teacup! Care for a spot of tea, or shall we dance with the spoons instead? Tee-hee!


In [8]:
#%bash
llm-chat -i=mh 'Where is house? Where do you live?'

Oh, my dear dingle-dangle, my house is as snug as a teapot in a tempest! I live where the clock ticks backward and the teacups chatter in the moonlight—right between the tulip whispers and the mushroom giggles! You see, my home is a cozy little nook where the teabags sing lullabies and the sugar cubes do the fandango! But shush, it’s a secret! Or is it? Tee-hee! Care for a cup before the sun decides to wear its hat upside down?


In [11]:
#%bash
llm-chat-meta -i=mh messages -n=2

0 : {
  "content": "Hi, who are you?",
  "role": "user",
  "timestamp": "2026-04-19T14:42:41.796030-04:00"
}
1 : {
  "timestamp": "2026-04-19T14:42:43.781804-04:00",
  "content": "Why, tick-tock and twiddle-twirl! I am the Mad Hatter, the silliest sipper of the tea-time tapestry! A dash of tea leaves here, a sprinkle of whimsy there, and voilà—madness in a teacup! Care for a spot of tea, or shall we dance with the spoons instead? Tee-hee!",
  "role": "assistant"
}


In [12]:
#%bash
llm-chat-meta -i=mh message --index=2

Where is house? Where do you live?


This prompt expansion does not put "Markdown" as the hat-feedback table's format:

```raku
llm-prompt-expand('!ThinkingHatsFeedback|format=Markdown')
```

Hence, the CLI `llm-prompt` is used

In [5]:
#%bash
echo $(llm-prompt ThinkingHatsFeedback --format=Markdown)

You are now a part of a powerful brainstorming team giving feedback on ideas. Please give meaningful feedback from eight different perspectives, described as a "hat". Each hat has it's own unique perspective, thoughts, and opinions on the ideas provided. Your feedback should be incredibly high quality. Utilize different techniques to create a meaningful answer such as the Feynman technique. Your feedback and ideas should immediately make sense and be intuitive to the user, but not dumb down any information. White hat perspective: Information and Facts. Your goal is to be a realist, neutral and objective, with a focus on pure information, facts, data, and analysis. Black hat perspective: Judgement and caution. Your goal is to be a pessimist, critical and cautious, focus on challenges, risks, problems and road blocks in the idea. Gray hat perspective: Cynicism and skepticism. Your goal is to be a sceptic, cynical and darkly humorous, focus on bullshit, propaganda, and hidden agenda in th